Problems: need to strip the start of name (has professor and stuff)
some profs dont exist? but ill still include (its Dr Chris Bell)
Some people have the title to not actually be their prof level, instead its in their name



In [89]:
#pip install requests beautifulsoup4

In [90]:
import requests

headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(url, headers=headers, timeout=10)

In [91]:
print(response.status_code)   # want 200
print(len(response.text))     # want something large, ~50k+ for these pages

403
919


In [92]:
with open("page.html", "w", encoding="utf-8") as f:
    f.write(response.text)

In [93]:
from bs4 import BeautifulSoup
soup = BeautifulSoup(response.text, "html.parser")
cards = soup.select(".person--teaser")
print(len(cards))

0


In [94]:
print(cards[0].prettify())

IndexError: list index out of range

In [ ]:
print(cards[0].select_one(".person__display-name a").get_text(strip=True))

Mr Ankit Jain


In [ ]:
print(cards[0].select_one(".position__title").get_text(strip=True))

In [ ]:
import re

PREFIX = re.compile(
    r"^(Associate Professor|Emeritus Professor|Professor|Dr|Mr|Mrs|Ms|Miss|A/Prof|Prof|Assoc\.? Prof\.?)\.?\s+",
    re.IGNORECASE
)



In [ ]:
import requests, time
from bs4 import BeautifulSoup
from urllib.parse import urljoin

import re

PREFIX = re.compile(
    r"^(Associate Professor|Emeritus Professor|Professor|Dr|Mr|Mrs|Ms|Miss|A/Prof|Prof|Assoc\.? Prof\.?)\.?\s+",
    re.IGNORECASE
)


LADDER = [
    ("Emeritus Professor",  r"emeritus prof"),
    ("Associate Professor", r"associate prof|a/prof"),
    ("Associate Lecturer",  r"associate lecturer"),
    ("Senior Lecturer",     r"senior lecturer"),
    ("Senior Research Fellow", r"senior research fellow"),
    ("Research Fellow",     r"research fellow"),
    ("Teaching Associate",  r"teaching associate"),
    ("Professor",           r"\bprofessor\b|chair in"),
    ("Lecturer",            r"\blecturer\b"),
]

def rank(title, prefix):
    for label, pat in LADDER:
        if title and re.search(pat, title, re.I):
            return label
    if prefix and prefix.lower() not in {"dr", "mr", "mrs", "ms", "miss"}:
        return prefix
    return None


def alive(url):
    try:
        return requests.head(url, allow_redirects=True, timeout=10).status_code == 200
    except requests.RequestException:
        return False


# loop 1 — extraction only, no network

# TARGETS = [
#     ("https://business.uq.edu.au/team/finance-discipline", "University of Queensland", "Finance"),
#     ("https://business.uq.edu.au/team/accounting-discipline", "University of Queensland", "Accounting")
# ]

TARGETS = [
    ("https://business.uq.edu.au/team/finance-discipline", "University of Queensland", "Finance")
]

records = []
for url, uni, disc in TARGETS:
    resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")
    cards = soup.select(".person--teaser")
    print(f"{disc}: {len(cards)} cards")
    for card in cards:
        link = card.select_one(".person__display-name a")
        if not link:
            continue
        name = link.get_text(strip=True)
        m = PREFIX.match(name)
        href = link["href"]

        titles = [t.get_text(strip=True) for t in card.select(".position__title")]
        titles = [t for t in titles if t]
        substantive = [t for t in titles if not t.startswith("Affiliate")]
        title = (substantive or titles or [None])[0]

        records.append({
            "university": uni,
            "discipline": disc,
            "name": name,
            "name_clean": PREFIX.sub("", name).strip(),
            "prefix": m.group(1) if m else None,
            "title": title,
            "title_clean": rank(title, m.group(1) if m else None),
            "profile_url": urljoin(url, href),
        })

    print(len(records))  # expect 24

# loop 2 — network checks
for r in records:
    r["alive"] = alive(r["profile_url"])
    time.sleep(1)

Finance: 24 cards
24


In [ ]:
for r in records:
    print(f"{r['title_clean']!s:25} <- {r['title']}")

Associate Lecturer        <- Associate Lecturer
None                      <- None
Professor                 <- Professor
Associate Lecturer        <- Associate Lecturer (Finance)
Teaching Associate        <- Teaching Associate
Senior Lecturer           <- Senior Lecturer
Senior Lecturer           <- Senior Lecturer
Professor                 <- Malcolm Broomhead Chair in Finance & Program Convenor (Bachelor of Advanced Finance and Economics) of UQ Business School
Senior Lecturer           <- Senior Lecturer in Finance
Lecturer                  <- Lecturer in Finance
Senior Lecturer           <- Senior Lecturer
Lecturer                  <- Lecturer
Lecturer                  <- Lecturer in Finance
Associate Professor       <- Discipline Convenor, Finance
Senior Lecturer           <- Senior Lecturer in Finance
Senior Lecturer           <- Senior Lecturer in Finance
Associate Professor       <- Associate Professor & Program Convenor (Bachelor of Commerce) of UQ Business School & Program Con

In [ ]:
KEEP = ["university", "discipline", "name_clean", "title_clean", "profile_url"]
records = [{k: r[k] for k in KEEP} for r in records]
records

[{'university': 'University of Queensland',
  'discipline': 'Finance',
  'name_clean': 'Jon Aster',
  'title_clean': 'Associate Lecturer',
  'profile_url': 'https://business.uq.edu.au/profile/9542/jon-aster'},
 {'university': 'University of Queensland',
  'discipline': 'Finance',
  'name_clean': 'Chris Bell',
  'title_clean': None,
  'profile_url': 'https://business.uq.edu.au/profile/17730/chris-bell'},
 {'university': 'University of Queensland',
  'discipline': 'Finance',
  'name_clean': 'Shaun Bond',
  'title_clean': 'Professor',
  'profile_url': 'https://business.uq.edu.au/profile/6239/shaun-bond'},
 {'university': 'University of Queensland',
  'discipline': 'Finance',
  'name_clean': 'Alexander Cameron',
  'title_clean': 'Associate Lecturer',
  'profile_url': 'https://business.uq.edu.au/profile/18606/alexander-cameron'},
 {'university': 'University of Queensland',
  'discipline': 'Finance',
  'name_clean': 'Yong Ming Chen',
  'title_clean': 'Teaching Associate',
  'profile_url': 'h

In [ ]:
#I dont know what Stephen Gray

In [ ]:
API_URL = "https://api.library.uq.edu.au/v1/records/search?export_to=&page=1&per_page=100&sort=published_date&order_by=desc&mode=advanced&key%5Brek_author_id%5D=533"

HEADERS = {
    "accept": "application/json, text/plain, */*",
    "accept-language": "en-US,en-AU;q=0.9,en;q=0.8",
    "origin": "https://espace.library.uq.edu.au",
    "referer": "https://espace.library.uq.edu.au/",
    "sec-fetch-dest": "empty",
    "sec-fetch-mode": "cors",
    "sec-fetch-site": "same-site",
    "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/150.0.0.0 Safari/537.36",
}

r = requests.get(API_URL, headers=HEADERS, timeout=10)
print(r.status_code, r.text[:300])

200 {"total":70,"took":50,"per_page":100,"current_page":1,"from":1,"to":70,"data":[{"rek_pid":"UQ:092048b","rek_title_xsdmf_id":null,"rek_title":"Sampling error and the joint estimation of imputation credit value and cash dividend value","rek_description_xsdmf_id":null,"rek_description":"The value of im


In [ ]:
import json
d = r.json()
print(d["total"], d["per_page"], d["current_page"])
print(json.dumps(d["data"][0], indent=2))

70 100 1
{
  "rek_pid": "UQ:092048b",
  "rek_title_xsdmf_id": null,
  "rek_title": "Sampling error and the joint estimation of imputation credit value and cash dividend value",
  "rek_description_xsdmf_id": null,
  "rek_description": "The value of imputation credits can only be estimated jointly with the value of cash dividends. We show that random variation across samples leads to estimates of credit value that move in the opposite direction to estimates of cash value. Derivative prices suggest a value for credits of 0.01 to 0.20 (0.01 to 0.07 if cash is worth 0.94, and 0.13 to 0.20 if cash is worth 0.87). Ex-dividend prices suggest a value for credits of 0.23 to 0.46 (0.23 to 0.36 if cash is worth 0.85, and 0.33 to 0.46 if cash is worth 0.75).",
  "rek_display_type_xsdmf_id": null,
  "rek_display_type": 179,
  "rek_status_xsdmf_id": null,
  "rek_status": 2,
  "rek_date_xsdmf_id": null,
  "rek_date": "2023-04-01T00:00:00Z",
  "rek_object_type_xsdmf_id": null,
  "rek_object_type": 3,
 

In [ ]:
rec = r.json()["data"][0]
print([k for k in rec if "journal" in k.lower()])

['fez_record_search_key_journal_name', 'fez_record_search_key_language_of_journal_name', 'fez_record_search_key_native_script_journal_name', 'fez_record_search_key_roman_script_journal_name', 'fez_record_search_key_translated_journal_name', 'fez_matched_journals']


In [ ]:
def parse(rec):
    jn = rec.get("fez_record_search_key_journal_name")
    return {
        "title": rec["rek_title"],
        "year": rec["rek_date"][:4] if rec.get("rek_date") else None,
        "type": rec.get("rek_genre"),
        "journal": jn,          # adjust once you see the shape
        "link": f"https://espace.library.uq.edu.au/view/{rec['rek_pid']}",
    }

In [ ]:
print(rec["fez_record_search_key_journal_name"])

{'rek_journal_name_id': 5458025, 'rek_journal_name_pid': 'UQ:092048b', 'rek_journal_name_xsdmf_id': None, 'rek_journal_name': 'Accounting and Finance'}


In [ ]:
for p in records:
    resp = requests.get(p["profile_url"], headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
    soup = BeautifulSoup(resp.text, "html.parser")
    link = soup.select_one('a[href*="author_id"]')
    p["espace_id"] = link["href"].rstrip("/").split("/")[-1] if link else None
    time.sleep(1)

print(sum(1 for p in records if p["espace_id"]), "of", len(records))

22 of 24


In [ ]:
[p["name_clean"] for p in records if not p["espace_id"]]

['Jon Aster', 'Chris Bell']

In [ ]:
print(len(records), sum(1 for p in records if p["espace_id"]))

24 22


In [ ]:
BASE = "https://api.library.uq.edu.au/v1/records/search"

pubs = []
for i, p in enumerate(records, 1):
    if not p["espace_id"]:
        print(f"{i} {p['name_clean']}: no eSpace ID, skipped")
        continue

    page, fetched, total = 1, 0, None
    while True:
        url = (f"{BASE}?export_to=&page={page}&per_page=100&sort=published_date"
               f"&order_by=desc&mode=advanced&key%5Brek_author_id%5D={p['espace_id']}")
        d = requests.get(url, headers=HEADERS, timeout=15).json()
        total = d["total"]

        for rec in d["data"]:
            jn = rec.get("fez_record_search_key_journal_name")

            mj = rec.get("fez_matched_journals") or []
            mj = mj[0] if isinstance(mj, list) and mj else (mj if isinstance(mj, dict) else None)
            fj = (mj or {}).get("fez_journal") or {}


            pubs.append({
                "name": p["name_clean"],
                "espace_id": p["espace_id"],
                "title": rec.get("rek_title"),
                "year": rec["rek_date"][:4] if rec.get("rek_date") else None,
                "type": rec.get("rek_genre"),
                "journal": jn["rek_journal_name"] if jn else None,
                "journal_id": fj.get("jnl_jid"),
                "era_id": fj.get("jnl_source_id_era"),
                "journal_canonical": fj.get("jnl_title"),
                "publisher": fj.get("jnl_publisher"),
                "link": f"https://espace.library.uq.edu.au/view/{rec['rek_pid']}",
            })

        fetched += len(d["data"])
        if fetched >= total or not d["data"]:
            break
        page += 1
        time.sleep(1)

    flag = "" if fetched == total else "  <-- MISMATCH"
    print(f"{i} {p['name_clean']}: {total} total, {fetched} fetched{flag}")
    time.sleep(1)

print(len(pubs), "publications from", len({x['name'] for x in pubs}), "people")

1 Jon Aster: no eSpace ID, skipped
2 Chris Bell: no eSpace ID, skipped
3 Shaun Bond: 36 total, 36 fetched
4 Alexander Cameron: 0 total, 0 fetched
5 Yong Ming Chen: 2 total, 2 fetched
6 Hasibul Chowdhury: 33 total, 33 fetched
7 Nicolas Eugster: 12 total, 12 fetched
8 Stephen Gray: 70 total, 70 fetched
9 Khoa Hoang: 12 total, 12 fetched
10 Weiting Hu: 2 total, 2 fetched
11 Ronghong Huang: 9 total, 9 fetched
12 Shirina Lin: 1 total, 1 fetched
13 Leo Luong: 10 total, 10 fetched
14 Jacquelyn Humphrey: 37 total, 37 fetched
15 Lin Mi: 15 total, 15 fetched
16 Suman Neupane-Joshi: 28 total, 28 fetched
17 Lily Nguyen: 22 total, 22 fetched
18 Vanitha Ragunathan: 20 total, 20 fetched
19 Dewan Rahman: 16 total, 16 fetched
20 Saphira Rekker: 40 total, 40 fetched
21 Eric Tan: 16 total, 16 fetched
22 Kelvin Tan: 31 total, 31 fetched
23 Elizabeth Zhu: 29 total, 29 fetched
24 Min Zhu: 36 total, 36 fetched
477 publications from 21 people


In [ ]:
from collections import Counter
print(Counter(x["type"] for x in pubs))
print(sum(1 for x in pubs if not x["journal"]), "with no journal")
print(sum(1 for x in pubs if not x["year"]), "with no year")
print(Counter(x["name"] for x in pubs).most_common())

Counter({'Journal Article': 392, 'Conference Paper': 19, 'Research Report': 12, 'Thesis': 11, 'Working Paper': 11, 'Book Chapter': 10, 'Data Collection': 7, 'Newspaper Article': 7, 'Book': 3, 'Generic Document': 2, 'Creative Work': 2, 'Conference Proceedings': 1})
82 with no journal
0 with no year
[('Stephen Gray', 70), ('Saphira Rekker', 40), ('Jacquelyn Humphrey', 37), ('Shaun Bond', 36), ('Min Zhu', 36), ('Hasibul Chowdhury', 33), ('Kelvin Tan', 31), ('Elizabeth Zhu', 29), ('Suman Neupane-Joshi', 28), ('Lily Nguyen', 22), ('Vanitha Ragunathan', 20), ('Dewan Rahman', 16), ('Eric Tan', 16), ('Lin Mi', 15), ('Nicolas Eugster', 12), ('Khoa Hoang', 12), ('Leo Luong', 10), ('Ronghong Huang', 9), ('Yong Ming Chen', 2), ('Weiting Hu', 2), ('Shirina Lin', 1)]


In [ ]:
print(sum(1 for x in pubs if x["type"] == "Journal Article" and not x["journal"]))

0


In [ ]:
from collections import Counter
print(Counter(x["type"] for x in pubs if not x["journal"]))

Counter({'Conference Paper': 16, 'Research Report': 12, 'Thesis': 11, 'Working Paper': 11, 'Book Chapter': 10, 'Data Collection': 7, 'Newspaper Article': 7, 'Book': 3, 'Generic Document': 2, 'Creative Work': 2, 'Conference Proceedings': 1})


In [ ]:
no_journal = [x for x in pubs if not x["journal"]]
print(len(no_journal))

import json
print(json.dumps(no_journal[0], indent=2))

82
{
  "name": "Shaun Bond",
  "espace_id": "6079108",
  "title": "Building business resilience",
  "year": "2022",
  "type": "Research Report",
  "journal": null,
  "link": "https://espace.library.uq.edu.au/view/UQ:e5f0850"
}


In [ ]:
pubs

[{'name': 'Shaun Bond',
  'espace_id': '6079108',
  'title': 'Mandatory disclosures of greenhouse gas emissions and managerial myopia: evidence from the United States',
  'year': '2025',
  'type': 'Journal Article',
  'journal': 'The International Journal of Accounting',
  'link': 'https://espace.library.uq.edu.au/view/UQ:02e94da'},
 {'name': 'Shaun Bond',
  'espace_id': '6079108',
  'title': 'Large language models and financial market sentiment',
  'year': '2023',
  'type': 'Journal Article',
  'journal': 'SSRN Electronic Journal',
  'link': 'https://espace.library.uq.edu.au/view/UQ:18ec8d4'},
 {'name': 'Shaun Bond',
  'espace_id': '6079108',
  'title': 'Seasonal patterns of earnings releases and post-earnings announcement drift',
  'year': '2023',
  'type': 'Journal Article',
  'journal': 'The Quarterly Review of Economics and Finance',
  'link': 'https://espace.library.uq.edu.au/view/UQ:4a30105'},
 {'name': 'Shaun Bond',
  'espace_id': '6079108',
  'title': 'Homemade equity offering

In [ ]:
print(rec.get("fez_matched_journals"))

{'mtj_pid': 'UQ:e97533a', 'mtj_jnl_id': 12029, 'mtj_status': 'A', 'fez_journal': {'jnl_jid': 12029, 'jnl_title': 'International Journal of Managerial Finance', 'jnl_abbrev_title': None, 'jnl_publisher': 'Emerald Publishing Limited', 'jnl_start_year': None, 'jnl_frequency': None, 'jnl_description': None, 'jnl_formats': None, 'jnl_is_refereed': None, 'jnl_source_id_era': '19630', 'jnl_advisory_statement': None, 'jnl_editing_user': None, 'jnl_editing_start_date': None, 'jnl_changeable_by_external_sources': True, 'jnl_created_date': '2021-02-22 01:33:31', 'jnl_updated_date': '2021-02-22 21:16:07', 'jnl_advisory_statement_type': None, 'fez_journal_era': [{'jnl_era_id': 103218, 'jnl_era_source_id': '19630', 'jnl_era_title': 'International Journal of Managerial Finance', 'jnl_era_source_year': 2023}, {'jnl_era_id': 12029, 'jnl_era_source_id': '19630', 'jnl_era_title': 'International Journal of Managerial Finance', 'jnl_era_source_year': 2018}, {'jnl_era_id': 32698, 'jnl_era_source_id': '19630

In [ ]:
arts = [x for x in pubs if x["type"] == "Journal Article"]
print(sum(1 for x in arts if x["era_id"]), "of", len(arts), "have an ERA id")

KeyError: 'era_id'